# Benchmark
Comparison of performance and accuracy between the methods in the `numerical_methods` library.

## Cell 1 – Imports

In [1]:
import numpy as np
import sympy as sp
from time import perf_counter
from scipy.integrate import quad, dblquad
from scipy.optimize._numdiff import approx_derivative
from scipy.optimize import brentq, approx_fprime

# Imported library already installed with pip.
import numerical_methods as nm

## Cell 2 – Helpers
Every section below times a method, compares it to a reference value, and prints one row.
These three functions do that job once so it isn't rewritten in each section.

In [2]:
def timed(func, *args, **kwargs):
    """Run func and return (result, elapsed_time_ms)."""
    start = perf_counter()
    result = func(*args, **kwargs)
    elapsed = (perf_counter() - start) * 1000
    return result, elapsed


def print_header(value_label="Result"):
    print(f"{'Method':30}{value_label:>15}{'Error':>15}{'Time (ms)':>15}")
    print("-" * 75)


def print_row(name, value, error, elapsed):
    """Print one formatted benchmark row. Pass value=None to show '—'."""
    value_str = f"{value:15.6f}" if value is not None else f"{'—':>15}"
    print(f"{name:30}{value_str}{error:15.2e}{elapsed:15.4f}")

## Cell 3 – Available methods
Only integration and differentiation methods are listed as dictionaries here: every function in each of these two families shares the same call signature (`method(f, a, b, n)` and `method(f, x, h)`), so a single loop can call any of them.

Root finding, series approximation, and linear algebra methods each take a different combination of arguments, so a shared dictionary would need one-off wrapping anyway — those calls are defined locally in their own cells, right next to the parameters they use.

In [3]:
integration_methods = {
    "Rectangle Rule": nm.rectangle_integrate,
    "Midpoint Method": nm.midpoint_integrate,
    "Trapezoid Rule": nm.trapezoidal_integrate,
    "First Simpson Rule": nm.simpson1_integrate,
    "Second Simpson Rule": nm.simpson2_integrate,
    "Gauss-Legendre Quadrature": nm.gauss_legendre_integrate,
    "Monte Carlo": nm.monte_carlo_integrate,
}

differentiation_methods = {
    "Forward Difference": nm.fd_forward_derivative,
    "Backward Difference": nm.fd_backward_derivative,
    "Central Difference": nm.fd_central_derivative,
    "Central Difference nth": nm.fd_nth_derivative,
    "Richardson Method": nm.richardson_derivative,
}

## Function of a variable 'f' that will be used throughout the benchmark.

In [4]:
f = lambda x: np.cos(x) ** 2 + np.sin(2 * x)

## Cell 4 – Numerical Integration

In [5]:
# Simple Integral
a, b = -np.pi, np.pi
n = 120  # interval subdivisions (sample count for Monte Carlo)
exact = np.pi

print_header()

for name, method in integration_methods.items():
    result, elapsed = timed(method, f, a, b, n)
    print_row(name, result, nm.error_calculate(exact, result), elapsed)

reference, elapsed_ref = timed(lambda: quad(f, a, b)[0])
print_row("SciPy (quad)", reference, nm.error_calculate(exact, reference), elapsed_ref)

print("")

# Double Integral
nx = ny = 1024
F = lambda x, y: x ** 2 * y
exact = 2.0 / 3.0

print_header()

result1, elapsed1 = timed(nm.midpoint_double_integrate, F, 0, 1, 0, 2, nx, ny)
result2, elapsed2 = timed(nm.trapezoidal_double_integrate, F, 0, 1, 0, 2, nx, ny)

reference, elapsed_ref = timed(lambda: dblquad(lambda y, x: F(x, y), 0, 1, lambda x: 0, lambda x: 2)[0])

print_row("Midpoint Double", result1, nm.error_calculate(exact, result1), elapsed1)
print_row("Trapezoidal Double", result2, nm.error_calculate(exact, result2), elapsed2)
print_row("SciPy (dblquad)", reference, nm.error_calculate(exact, reference), elapsed_ref)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Rectangle Rule                       3.141593       2.83e-16         0.1318
Midpoint Method                      3.141593       0.00e+00         0.0947
Trapezoid Rule                       3.141593       0.00e+00         0.2417
First Simpson Rule                   3.141593       0.00e+00         0.1253
Second Simpson Rule                  3.141593       0.00e+00         0.1024
Gauss-Legendre Quadrature            3.141593       6.50e-15         3.1165
Monte Carlo                          2.810407       1.05e-01         0.1866
SciPy (quad)                         3.141593       1.41e-16         0.0579

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Midpoint Double                      0.666667       2.38e-07       240.8002
Trapezoidal

## Cell 5 – Numerical Differentiation

In [6]:
evaluation_point = 2
delta_x = 0.0001

x_sym = sp.symbols('x')
f_sym = sp.cos(x_sym) ** 2 + sp.sin(2 * x_sym)
df_sym = sp.diff(f_sym, x_sym)
reference = float(df_sym.subs(x_sym, evaluation_point))

print_header()

for name, method in differentiation_methods.items():
    result, elapsed = timed(method, f, evaluation_point, delta_x)
    print_row(name, result, nm.error_calculate(reference, result), elapsed)

# SciPy: approx_fprime(x, f, h) -> gradient
result, elapsed = timed(approx_fprime, np.array([evaluation_point]), lambda v: f(v[0]), delta_x)
print_row("SciPy (approx_fprime)", result[0], nm.error_calculate(result[0], reference), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Forward Difference                  -0.550268       3.94e-04         0.0417
Backward Difference                 -0.550701       3.94e-04         0.0056
Central Difference                  -0.550485       6.67e-09         0.0023
Central Difference nth              -0.550485       1.67e-09         0.0083
Richardson Method                   -0.550485       5.03e-13         0.0053
SciPy (approx_fprime)               -0.550268       3.94e-04         0.2895


## Cell 6 – Root Finding

In [7]:
root_methods = [
    ("Bisection Method", lambda: nm.bisection_calculate(f, *root_interval, tol)),
    ("Newton-Raphson Method", lambda: nm.newton_raphson_calculate(f, x0_root, max_iter, tol)[0]),
    ("Ridders Method", lambda: nm.ridders_calculate(f, *root_interval, max_iter, tol)[0]),
    ("Brent Method (SciPy)", lambda: brentq(f, *root_interval, xtol=tol)),
]

# f has a sign change between -0.5 and -0.4 (bracket for Bisection / Ridders)
root_interval = (-0.5, -0.4)
x0_root = -0.45
max_iter = 100
tol = 1e-6

reference = float(sp.nsolve(f_sym, x_sym, x0_root))

print_header()

for name, method in root_methods:
    root, elapsed = timed(method)
    print_row(name, root, nm.error_calculate(reference, root), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Bisection Method                    -0.463648       5.03e-07         0.0485
Newton-Raphson Method               -0.463648       8.45e-09         0.0183
Ridders Method                      -0.463648       3.91e-09         0.0155
Brent Method (SciPy)                -0.463648       3.42e-09         0.0480


## Cell 7 – Series Approximation

In [8]:
series_methods = [
    ("Taylor Series", lambda: nm.taylor_approx(f, evaluation_x, expansion_point, order)),
    ("Fourier Series", lambda: nm.fourier_approx(f, evaluation_x, b, order)),
]

expansion_point = 0
order = 6
evaluation_x = 1.0
reference = f(evaluation_x)

print_header()
for name, method in series_methods:
    result, elapsed = timed(method)
    print_row(name, result, nm.error_calculate(reference, result), elapsed)

Method                                 Result          Error      Time (ms)
---------------------------------------------------------------------------
Taylor Series                        1.222223       1.75e-02         0.1105
Fourier Series                       1.201224       1.85e-16         1.7437


## Cell 8 – Linear Algebra

In [9]:
A = np.array([
    [10., 2., 1., 3., 0.],
    [2., 12., 2., 1., 4.],
    [1., 2., 15., 3., 2.],
    [3., 1., 3., 14., 5.],
    [0., 4., 2., 5., 13.]
])

b_vec = np.array([15., 20., 30., 25., 18.])

reference_solution = np.linalg.solve(A, b_vec)
reference_det = np.linalg.det(A)

# --- Linear system solve: Ax = b ---
solvers = [
    ("Linear System (Gauss)", lambda: nm.linearsystem_solve(A, b_vec, method="gauss")),
    ("Linear System (LU)", lambda: nm.linearsystem_solve(A, b_vec, method="lu")),
    ("Linear System (Cholesky)", lambda: nm.linearsystem_solve(A, b_vec, method="cholesky")),
    ("Linear System (QR)", lambda: nm.linearsystem_solve(A, b_vec, method="QR")),
    ("Linear System (NumPy)", lambda: np.linalg.solve(A, b_vec)),
]

print_header("Result (norm)")
for name, solver in solvers:
    x, elapsed = timed(solver)
    error = np.linalg.norm(x - reference_solution)
    print_row(name, np.linalg.norm(x), error, elapsed)

print("-" * 75)

# --- LU factorization: A = LU ---
(L, U), elapsed = timed(nm.lu_decomposition, A)
reconstruction_error = np.linalg.norm(L @ U - A)
print_row("LU Decomposition", None, reconstruction_error, elapsed)

print("-" * 75)

# --- Determinant ---
determinants = [
    ("Determinant (Gauss)", lambda: nm.determinant_calculate(A, method="gauss")),
    ("Determinant (LU)", lambda: nm.determinant_calculate(A, method="lu")),
    ("Determinant (NumPy)", lambda: np.linalg.det(A)),
]

for name, det_func in determinants:
    det, elapsed = timed(det_func)
    print_row(name, det, abs(det - reference_det), elapsed)

print("-" * 75)

# --- Jacobian ---
def F(v):
    x_, y_ = v
    return np.array([x_**2 + y_**2 - 4, x_ - y_])

point = np.array([1.0, 1.0])
h = 1e-6

# Separate symbols from the x_sym used in Cells 5-6 (scalar case) to avoid overwriting them.
x_j, y_j = sp.symbols('x y')
F_sym = sp.Matrix([x_j**2 + y_j**2 - 4, x_j - y_j])
reference_jacobian = np.array(
    F_sym.jacobian([x_j, y_j]).subs({x_j: point[0], y_j: point[1]})
).astype(float)

jacobians = [
    ("Jacobian", lambda: nm.jacobian_calculate(F, point, h)),
    ("Jacobian (SciPy)", lambda: approx_derivative(F, point)),
]

for name, jac_func in jacobians:
    J, elapsed = timed(jac_func)
    error = np.linalg.norm(J - reference_jacobian)
    print_row(name, None, error, elapsed)
    print(J)

Method                          Result (norm)          Error      Time (ms)
---------------------------------------------------------------------------
Linear System (Gauss)                2.329031       4.58e-16         0.2126
Linear System (LU)                   2.329031       3.85e-16         0.0929
Linear System (Cholesky)             2.329031       2.29e-16         0.2492
Linear System (QR)                   2.329031       1.12e-15         0.2045
Linear System (NumPy)                2.329031       0.00e+00         0.0535
---------------------------------------------------------------------------
LU Decomposition                            —       4.44e-16         0.0551
---------------------------------------------------------------------------
Determinant (Gauss)             209655.000000       4.07e-10         0.0823
Determinant (LU)                209655.000000       3.78e-10         0.0683
Determinant (NumPy)             209655.000000       0.00e+00         0.0450
------------